In [ ]:
# ## Import libraries
import os
import numpy as np
import rasterio
from pathlib import Path

In [ ]:
# ## Set up variables
DIR_IMG_BASE = Path("2_sat_imgs")
DIR_MSK_BASE = Path("3_groundtruth")

DIR_IMG_TRAIN = DIR_IMG_BASE / "train_valid"
DIR_IMG_TEST = DIR_IMG_BASE / "test_valid"
DIR_MSK_TRAIN = DIR_MSK_BASE / "gt_train_valid"
DIR_MSK_TEST = DIR_MSK_BASE / "gt_test_valid"

DIR_OUT_BASE = Path("4_unet_input")

DIR_OUT_TRAIN = DIR_OUT_BASE / "train"
DIR_OUT_TEST = DIR_OUT_BASE / "test"

DIR_OUT_TRAIN_IMG = DIR_OUT_TRAIN / "images"
DIR_OUT_TRAIN_MSK_BI = DIR_OUT_TRAIN / "masks_binary"
DIR_OUT_TRAIN_MSK_MC = DIR_OUT_TRAIN / "masks_multiclass"
DIR_OUT_TEST_IMG = DIR_OUT_TEST / "images"
DIR_OUT_TEST_MSK_BI = DIR_OUT_TEST / "masks_binary"
DIR_OUT_TEST_MSK_MC = DIR_OUT_TEST / "masks_multiclass"

DIR_OUT_TRAIN_IMG.mkdir(parents=True, exist_ok=True)
DIR_OUT_TRAIN_MSK_BI.mkdir(parents=True, exist_ok=True)
DIR_OUT_TRAIN_MSK_MC.mkdir(parents=True, exist_ok=True)
DIR_OUT_TEST_IMG.mkdir(parents=True, exist_ok=True)
DIR_OUT_TEST_MSK_BI.mkdir(parents=True, exist_ok=True)
DIR_OUT_TEST_MSK_MC.mkdir(parents=True, exist_ok=True)

WETLAND_CLASSES = [1, 2, 3, 4, 5]
PATCH_SIZE = 256
CENTER_SIZE = 512


In [ ]:
# ## Set up functions
def load_tif(path: Path) -> tuple:
    with rasterio.open(path) as src:
        img = src.read()  # (bands, H, W)
        nodata = src.nodata
    return img, nodata

def center_crop(img, size=512):
    _, H, W = img.shape
    start_x = (W - size) // 2
    start_y = (H - size) // 2
    return img[:, start_y:start_y+size, start_x:start_x+size]

def split_into_4(img):
    patches = []
    patches.append(img[:, 0:256, 0:256])
    patches.append(img[:, 0:256, 256:512])
    patches.append(img[:, 256:512, 0:256])
    patches.append(img[:, 256:512, 256:512])
    return patches

def process_pair(img_path, mask_path, out_img_dir, out_mask_bin_dir, out_mask_multi_dir):
    img, _ = load_tif(img_path)
    mask, nodata = load_tif(mask_path)

    # Remove band dimension
    mask = mask[0]

    # ---------- Multi-class mask ----------
    multi_mask = mask.copy()

    if nodata is not None:
        multi_mask[mask == nodata] = 0  # set nodata to background

    multi_mask = multi_mask[np.newaxis, :, :]

    # ---------- Binary mask ----------
    binary_mask = np.isin(mask, WETLAND_CLASSES).astype(np.uint8)

    if nodata is not None:
        binary_mask[mask == nodata] = 0

    binary_mask = binary_mask[np.newaxis, :, :]

    # ---------- Crop ----------
    img = center_crop(img)
    binary_mask = center_crop(binary_mask)
    multi_mask = center_crop(multi_mask)

    # ---------- Split ----------
    img_patches = split_into_4(img)
    bin_patches = split_into_4(binary_mask)
    multi_patches = split_into_4(multi_mask)

    base_img = Path(img_path).stem
    base_msk = Path(mask_path).stem

    for i in range(4):
        np.save(os.path.join(out_img_dir, f"{base_img}_p{i}.npy"), img_patches[i])
        np.save(os.path.join(out_mask_bin_dir, f"{base_msk}_p{i}.npy"), bin_patches[i])
        np.save(os.path.join(out_mask_multi_dir, f"{base_msk}_p{i}.npy"), multi_patches[i])

def build_pairs(img_dir, mask_dir):
    img_files = sorted(img_dir.glob("*.tif"))
    pairs = []

    for img_path in img_files:
        base = img_path.stem  # train_001_001
        mask_name = base + "_gt.tif"  # train_001_001_gt.tif
        mask_path = mask_dir / mask_name

        if mask_path.exists():
            pairs.append((img_path, mask_path))
        else:
            print("Mask not found for:", img_path)

    return pairs

In [ ]:
# ## Run data pre-processing
# Make image-mask pairs
train_pairs = build_pairs(DIR_IMG_TRAIN, DIR_MSK_TRAIN)
test_pairs  = build_pairs(DIR_IMG_TEST, DIR_MSK_TEST)

print("Train pairs:", len(train_pairs))
print("Test pairs:", len(test_pairs))

# Divide them into 4 patches
for img_path, mask_path in train_pairs:
    process_pair(
        img_path=img_path,
        mask_path=mask_path,
        out_img_dir=DIR_OUT_TRAIN_IMG,
        out_mask_bin_dir=DIR_OUT_TRAIN_MSK_BI,
        out_mask_multi_dir=DIR_OUT_TRAIN_MSK_MC
    )

for img_path, mask_path in test_pairs:
    process_pair(
        img_path=img_path,
        mask_path=mask_path,
        out_img_dir=DIR_OUT_TEST_IMG,
        out_mask_bin_dir=DIR_OUT_TEST_MSK_BI,
        out_mask_multi_dir=DIR_OUT_TEST_MSK_MC
    )

print("Train image patches:", len(list(DIR_OUT_TRAIN_IMG.glob("*.npy"))))
print("Train mask (binary) patches:", len(list(DIR_OUT_TRAIN_MSK_BI.glob("*.npy"))))
print("Train mask (multiclass) patches:", len(list(DIR_OUT_TRAIN_MSK_MC.glob("*.npy"))))

print("Test image patches:", len(list(DIR_OUT_TEST_IMG.glob("*.npy"))))
print("Test mask (binary) patches:", len(list(DIR_OUT_TEST_MSK_BI.glob("*.npy"))))
print("Test mask (multiclass) patches:", len(list(DIR_OUT_TEST_MSK_MC.glob("*.npy"))))